# ANN vs Decision Tree: Prediksi Cooling Load
Dataset: Energy Efficiency UCI (ENB2012_data.xlsx)
Target: Y2 (Cooling Load)
Model: Neural Network (MLP via Keras) vs Decision Tree Regressor

## Bagian 0 — Install Library
Jalankan cell ini sekali saja untuk memastikan semua library tersedia.

In [ ]:
# Install library yang dibutuhkan
# Jalankan hanya jika belum terinstall di environment kamu
%pip install pandas numpy scikit-learn tensorflow openpyxl

## Bagian 1 — Import Library
Semua library yang digunakan di-import di sini.
- `pandas` dan `numpy`: manipulasi data
- `sklearn`: preprocessing, split data, model Decision Tree, dan metrik evaluasi
- `tensorflow.keras`: membangun model Neural Network (MLP)
- `time`: mengukur runtime training dan evaluasi setiap model

In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

## Bagian 2 — Load dan Eksplorasi Dataset
Dataset dibaca dari file `ENB2012_data.xlsx`.
- Kolom X1 sampai X8 adalah fitur input (karakteristik bangunan)
- Kolom Y2 adalah target yang ingin diprediksi: Cooling Load
- Kita cek jumlah baris, kolom, dan apakah ada missing value

In [ ]:
# Baca dataset dari file Excel
df = pd.read_excel('ENB2012_data.xlsx')

# Tampilkan 5 baris pertama untuk memastikan data terbaca dengan benar
print('5 baris pertama dataset:')
print(df.head())

# Cek ukuran dataset: jumlah baris dan kolom
print(f'\nUkuran dataset: {df.shape[0]} baris, {df.shape[1]} kolom')

# Cek apakah ada missing value di setiap kolom
print('\nJumlah missing value per kolom:')
print(df.isnull().sum())

# Tampilkan statistik deskriptif dataset
print('\nStatistik deskriptif:')
print(df.describe())

## Bagian 3 — Pisahkan Fitur dan Target
- `X`: semua kolom fitur input (X1 sampai X8)
- `y`: hanya kolom Y2 yang merupakan Cooling Load
- Kolom Y1 (Heating Load) tidak dipakai karena fokus tugas kita adalah Cooling Load

In [ ]:
# Pilih 8 kolom fitur sebagai input
feature_cols = ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']
X = df[feature_cols].values

# Pilih Y2 sebagai target (Cooling Load)
y = df['Y2'].values

print(f'Shape X (fitur): {X.shape}')
print(f'Shape y (target): {y.shape}')

## Bagian 4 — Train-Test Split dan Scaling
Data dibagi 80% untuk training dan 20% untuk testing.
- `random_state=42` memastikan pembagian data selalu sama setiap kali dijalankan
- `StandardScaler` digunakan untuk normalisasi fitur: rata-rata jadi 0, standar deviasi jadi 1
- Scaler di-fit **hanya** pada data training, lalu diterapkan ke training dan test
- Ini penting untuk menghindari *data leakage* (test set tidak boleh mempengaruhi proses scaling)

In [ ]:
# Bagi data jadi train (80%) dan test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Jumlah data training: {X_train.shape[0]}')
print(f'Jumlah data testing : {X_test.shape[0]}')

# Inisialisasi StandardScaler
scaler = StandardScaler()

# Fit scaler pada X_train, lalu transform X_train dan X_test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## Bagian 5 — Model Decision Tree (Pembanding)
Decision Tree Regressor digunakan sebagai model baseline yang sederhana.
- `max_depth=10` membatasi kedalaman pohon supaya tidak *overfitting*
- `random_state=42` memastikan hasil yang konsisten
- Runtime diukur dari sebelum training sampai setelah prediksi di test set
- Metrik yang dihitung: MSE, MAE, dan R2 Score

In [ ]:
# Inisialisasi model Decision Tree Regressor
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)

# Mulai timer sebelum training
dt_start = time.time()

# Training model menggunakan data training yang sudah di-scale
dt_model.fit(X_train_scaled, y_train)

# Prediksi pada data test
dt_pred = dt_model.predict(X_test_scaled)

# Stop timer setelah prediksi selesai
dt_end = time.time()
dt_runtime = dt_end - dt_start

# Hitung metrik evaluasi
dt_mse = mean_squared_error(y_test, dt_pred)
dt_mae = mean_absolute_error(y_test, dt_pred)
dt_r2  = r2_score(y_test, dt_pred)

print('=== Decision Tree Regressor ===')
print(f'MSE    : {dt_mse:.4f}')
print(f'MAE    : {dt_mae:.4f}')
print(f'R2     : {dt_r2:.4f}')
print(f'Runtime: {dt_runtime:.4f} detik')

## Bagian 6 — Model Neural Network MLP (Keras)
Model MLP dibangun menggunakan Keras Sequential.
- **Input layer**: otomatis menyesuaikan dengan 8 fitur
- **Hidden layer 1**: 64 neuron, fungsi aktivasi ReLU (menangkap hubungan non-linear)
- **Hidden layer 2**: 32 neuron, fungsi aktivasi ReLU
- **Output layer**: 1 neuron tanpa aktivasi (regresi, prediksi nilai kontinu)
- `optimizer='adam'`: adaptive learning rate, cocok untuk regresi
- `loss='mse'`: Mean Squared Error sebagai fungsi loss
- `EarlyStopping`: menghentikan training jika val_loss tidak membaik selama 10 epoch, mencegah overfitting
- Runtime diukur dari sebelum training sampai setelah prediksi

In [ ]:
# Bangun arsitektur model MLP
nn_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),  # Hidden layer 1
    Dense(32, activation='relu'),                                           # Hidden layer 2
    Dense(1)                                                                # Output layer (regresi)
])

# Compile model: tentukan optimizer, loss function
nn_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# EarlyStopping: hentikan training jika tidak ada perbaikan selama 10 epoch
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Mulai timer sebelum training
nn_start = time.time()

# Training model
history = nn_model.fit(
    X_train_scaled, y_train,
    epochs=200,           # maksimum 200 epoch
    batch_size=32,        # 32 sampel per batch
    validation_split=0.1, # 10% data train dipakai sebagai validasi selama training
    callbacks=[early_stop],
    verbose=0             # matikan output log per epoch supaya output lebih bersih
)

# Prediksi pada data test
nn_pred = nn_model.predict(X_test_scaled).flatten()

# Stop timer setelah prediksi selesai
nn_end = time.time()
nn_runtime = nn_end - nn_start

# Hitung metrik evaluasi
nn_mse = mean_squared_error(y_test, nn_pred)
nn_mae = mean_absolute_error(y_test, nn_pred)
nn_r2  = r2_score(y_test, nn_pred)

print('=== Neural Network MLP (Keras) ===')
print(f'MSE    : {nn_mse:.4f}')
print(f'MAE    : {nn_mae:.4f}')
print(f'R2     : {nn_r2:.4f}')
print(f'Runtime: {nn_runtime:.4f} detik')
print(f'Epochs yang dijalankan: {len(history.history["loss"])}')

## Bagian 7 — Perbandingan Hasil
Semua hasil metrik dari kedua model digabungkan ke dalam satu tabel.
Ini memudahkan analisis model mana yang lebih baik dari sisi akurasi dan kecepatan.

In [ ]:
# Buat tabel perbandingan hasil kedua model
results = pd.DataFrame({
    'Model'  : ['Decision Tree', 'Neural Network (MLP)'],
    'MSE'    : [round(dt_mse, 4), round(nn_mse, 4)],
    'MAE'    : [round(dt_mae, 4), round(nn_mae, 4)],
    'R2'     : [round(dt_r2, 4),  round(nn_r2, 4)],
    'Runtime (s)': [round(dt_runtime, 4), round(nn_runtime, 4)]
})

print('=== Perbandingan Performa Model ===')
print(results.to_string(index=False))

## Bagian 8 — Analisis dan Interpretasi
Tulis analisis kamu di sini berdasarkan hasil yang muncul di Bagian 7.
Beberapa poin yang bisa kamu bahas:
- Model mana yang menghasilkan MSE dan MAE lebih kecil?
- Model mana yang memiliki R2 lebih tinggi (lebih mendekati 1.0 berarti lebih baik)?
- Berapa selisih runtime antara Decision Tree dan Neural Network?
- Apakah kompleksitas NN sepadan dengan peningkatan performa dibanding Decision Tree?
- Mengapa NN lebih lambat? (karena proses backpropagation dan banyak epoch)

Contoh kalimat yang bisa kamu tulis (sesuaikan dengan angka aktual):
'Neural Network menghasilkan MSE lebih rendah dibanding Decision Tree, menunjukkan prediksi yang lebih akurat. Namun, runtime NN jauh lebih lama karena proses optimasi bobot melalui banyak epoch. Untuk dataset kecil seperti ini (768 data), Decision Tree sudah memberikan performa yang kompetitif dengan waktu komputasi yang jauh lebih singkat.'